# Lab 3 · Exercise B — ModelConfig and hypothesis test
The same top-quark model (Gaussian constraint), now analysed with **RooStats**.

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
def build_gauss(non, b0_val, sigma_b):
    """Counting model with a Gaussian constraint on b:  Pois(n_on | s+b) * Gauss(b0 | b, sigma_b)."""
    s  = ROOT.RooRealVar("s", "signal", max(non - b0_val, 0.5), 0, 200)
    b  = ROOT.RooRealVar("b", "background", b0_val, 0.01, 200)
    b0 = ROOT.RooRealVar("b0", "global observable", b0_val, 0, 200); b0.setConstant(True)
    sb = ROOT.RooRealVar("sb", "sigma_b", sigma_b);                  sb.setConstant(True)
    n_on = ROOT.RooRealVar("n_on", "n_on", 0, 300)

    mu_on  = ROOT.RooFormulaVar("mu_on", "@0+@1", ROOT.RooArgList(s, b))
    p_on   = ROOT.RooPoisson("p_on", "p_on", n_on, mu_on)
    constr = ROOT.RooGaussian("constr", "constr", b0, b, sb)
    pdf    = ROOT.RooProdPdf("pdf", "pdf", ROOT.RooArgList(p_on, constr))

    data = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on))   # only n_on: b0 is NOT in the data
    n_on.setVal(non)
    data.add(ROOT.RooArgSet(n_on))
    nll = pdf.createNLL(data, ROOT.RooFit.GlobalObservables(ROOT.RooArgSet(b0)))
    return dict(locals())

## B1 · Workspace and ModelConfig
Import model and data in a `RooWorkspace`, then define the **roles**: pdf, parameter of interest `s`, nuisance `b`,
observable `n_on`, global observable `b0`.

In [ ]:
non, noff, tau_val = 17, 40, 10.53
m = build_gauss(non, noff / tau_val, np.sqrt(noff) / tau_val)

w = ROOT.RooWorkspace("w")
w.Import(m["pdf"])
w.Import(m["data"])

mc = ROOT.RooStats.ModelConfig("mc", w)
# TODO: SetPdf (w.pdf("pdf")), SetParametersOfInterest (RooArgSet of w.var("s")),
#       SetNuisanceParameters (b), SetObservables (n_on), SetGlobalObservables (b0)
data = w.data("d")
mc.Print()

## B2 · Profile-likelihood intervals (68 % and 95 %)

In [ ]:
s = w.var("s")
w.pdf("pdf").fitTo(data, ROOT.RooFit.GlobalObservables(mc.GetGlobalObservables()), ROOT.RooFit.PrintLevel(-1))
s_hat = s.getVal()

plc = ROOT.RooStats.ProfileLikelihoodCalculator(data, mc)
for cl in (0.683, 0.95):
    plc.SetConfidenceLevel(cl)
    interval = plc.GetInterval()
    # TODO: print  cl, s_hat, interval.LowerLimit(s), interval.UpperLimit(s)

In [ ]:
check("intervals computed", len(globals().get("res_int", {})) == 2)
check("68 % interval close to [9.4, 17.7]", abs(res_int[0.683][0] - 9.35) < 0.5 and abs(res_int[0.683][1] - 17.7) < 0.5)
check("95 % wider than 68 %", res_int[0.95][1] - res_int[0.95][0] > res_int[0.683][1] - res_int[0.683][0])

## B3 · Significance of the background-only hypothesis

In [ ]:
# Null hypothesis: s = 0
s_null = ROOT.RooRealVar("s", "s", 0)
plc.SetNullParameters(ROOT.RooArgSet(s_null))
htr = None   # TODO: htr = plc.GetHypoTest()
# TODO: print htr.Significance() and htr.NullPValue()

In [ ]:
check("hypothesis test done", htr is not None)
check("significance about 4.6", abs(htr.Significance() - 4.6) < 0.3)

## B4 · Compare with the counting formulae
With $s=\\hat s$ and $b=\\hat b$, evaluate $s/\\sqrt b$ and the Asimov estimate
$Z_A=\\sqrt{2[(s+b)\\ln(1+s/b)-s]}$ (background known). Compare with the RooStats $Z$.

In [ ]:
s_v, b_v = 13.2, 3.8
Z_simple = None   # TODO: s / sqrt(b)
Z_asimov = None   # TODO: sqrt(2 * ((s+b) * ln(1 + s/b) - s))
print("s/sqrt(b) = %.2f   Z_A = %.2f   RooStats (b uncertain) = %.2f" % (Z_simple, Z_asimov, htr.Significance()))

In [ ]:
check("Z_A close to 4.95", abs(Z_asimov - 4.95) < 0.05)
check("s/sqrt(b) overestimates the significance most", Z_simple > Z_asimov > htr.Significance())

## Questions (2–3 lines each)
1. Why is $s/\\sqrt b$ so far from the RooStats result? When would it be a good approximation?
2. $Z_A$ is higher than the RooStats $Z$. What uncertainty does $Z_A$ ignore?
3. Which calculator would you use to quote a **discovery**, and which for an **upper limit** when $n_{\\rm on}$ is compatible with the background?